**Programación 2 - Actividad 2**

Nombre: Joseph Alexander Peña Aranda

Paralelo: "B"

2.**Situación a resolver**

Una biblioteca universitaria registra sus préstamos de forma manual. Necesita identificar qué materiales están disponibles, quién los tiene prestados y cuántos préstamos mantiene activos cada usuario.

Para solucionar esta necesidad se desarrollará un sistema utilizando Python y programación orientada a objetos, que permita registrar materiales, usuarios, préstamos y devoluciones.

El sistema contará con tres tipos de materiales: libros, revistas y audiolibros. También permitirá registrar estudiantes y docentes, considerando que cada uno tiene un límite diferente de préstamos activos.

La aplicación deberá mantener la información organizada y validar cada operación para evitar errores, como prestar un material que ya está ocupado o permitir que un usuario supere su límite de préstamos.

3.**Alcance y recursos**

La solución se desarrollará en Google Colab utilizando el lenguaje de programación Python y el paradigma de programación orientada a objetos.

El sistema permitirá realizar las operaciones mediante llamadas a métodos, sin necesidad de implementar una interfaz gráfica o un menú interactivo.

Se utilizarán clases, objetos, propiedades, herencia, encapsulamiento, polimorfismo, excepciones y métodos especiales.

Para el manejo de fechas se utilizarán date y timedelta del módulo datetime de Python.

La información permanecerá almacenada en memoria durante la ejecución del programa. Cada biblioteca tendrá sus propias colecciones de materiales, usuarios y préstamos.

4.**Jerarquía de materiales**


4.1 Clase base Material

In [1]:
from datetime import date, timedelta


class Material:
    def __init__(self, codigo, titulo):
        if not isinstance(codigo, str) or not isinstance(titulo, str):
            raise TypeError("El codigo y titulo deben ser textos")

        if not codigo.strip() or not titulo.strip():
            raise ValueError("El codigo y titulo no pueden estar vacios")

        self._codigo = codigo.strip().upper()
        self._titulo = titulo.strip()
        self._disponible = True

    @property
    def codigo(self):
        return self._codigo

    @property
    def titulo(self):
        return self._titulo

    @property
    def disponible(self):
        return self._disponible

    def prestar(self):
        if not self._disponible:
            raise ValueError("El material ya esta prestado")

        self._disponible = False

    def devolver(self):
        if self._disponible:
            raise ValueError("El material ya esta disponible")

        self._disponible = True

    def dias_prestamo(self):
        raise NotImplementedError(
            "Este metodo debe implementarse en las clases hijas"
        )

    def __str__(self):
        estado = "Disponible" if self.disponible else "Prestado"

        return (
            f"Codigo: {self.codigo} | "
            f"Titulo: {self.titulo} | "
            f"Estado: {estado}"
        )

4.2 Clases Libro, Revista y Audiolibro

In [2]:
class Libro(Material):
    def __init__(self, codigo, titulo, autor):
        super().__init__(codigo, titulo)

        if not isinstance(autor, str):
            raise TypeError("El autor debe ser un texto")

        if not autor.strip():
            raise ValueError("El autor no puede estar vacio")

        self.autor = autor.strip()

    def dias_prestamo(self):
        return 14

    def __str__(self):
        return (
            f"{super().__str__()} | "
            f"Autor: {self.autor} | "
            f"Plazo: 14 dias"
        )


class Revista(Material):
    def __init__(self, codigo, titulo, numero_edicion):
        super().__init__(codigo, titulo)

        if isinstance(numero_edicion, bool) or not isinstance(numero_edicion, int):
            raise TypeError("El numero de edicion debe ser un entero")

        if numero_edicion <= 0:
            raise ValueError("El numero de edicion debe ser positivo")

        self.numero_edicion = numero_edicion

    def dias_prestamo(self):
        return 7

    def __str__(self):
        return (
            f"{super().__str__()} | "
            f"Edicion: {self.numero_edicion} | "
            f"Plazo: 7 dias"
        )


class Audiolibro(Material):
    def __init__(self, codigo, titulo, duracion_minutos):
        super().__init__(codigo, titulo)

        if isinstance(duracion_minutos, bool) or not isinstance(
            duracion_minutos, (int, float)
        ):
            raise TypeError("La duracion debe ser numerica")

        if duracion_minutos <= 0:
            raise ValueError("La duracion debe ser positiva")

        self.duracion_minutos = duracion_minutos

    def dias_prestamo(self):
        return 10

    def __str__(self):
        return (
            f"{super().__str__()} | "
            f"Duracion: {self.duracion_minutos} minutos | "
            f"Plazo: 10 dias"
        )

5.**Jerarquía de usuarios**


5.1 Clase base Usuario

In [3]:
class Usuario:
    def __init__(self, identificacion, nombre):
        if not isinstance(identificacion, str) or not isinstance(nombre, str):
            raise TypeError("La identificacion y el nombre deben ser textos")

        if not identificacion.strip() or not nombre.strip():
            raise ValueError("La identificacion y el nombre no pueden estar vacios")

        self._identificacion = identificacion.strip()
        self.nombre = nombre.strip()

    @property
    def identificacion(self):
        return self._identificacion

    def limite_prestamos(self):
        raise NotImplementedError(
            "Este metodo debe implementarse en las clases hijas"
        )

    def __str__(self):
        return (
            f"Tipo: {type(self).__name__} | "
            f"Identificacion: {self.identificacion} | "
            f"Nombre: {self.nombre}"
        )

5.2 Clases Estudiante y Docente

In [4]:
class Estudiante(Usuario):
    def __init__(self, identificacion, nombre, carrera):
        super().__init__(identificacion, nombre)

        if not isinstance(carrera, str):
            raise TypeError("La carrera debe ser un texto")

        if not carrera.strip():
            raise ValueError("La carrera no puede estar vacia")

        self.carrera = carrera.strip()

    def limite_prestamos(self):
        return 3

    def __str__(self):
        return (
            f"{super().__str__()} | "
            f"Carrera: {self.carrera}"
        )


class Docente(Usuario):
    def __init__(self, identificacion, nombre, facultad):
        super().__init__(identificacion, nombre)

        if not isinstance(facultad, str):
            raise TypeError("La facultad debe ser un texto")

        if not facultad.strip():
            raise ValueError("La facultad no puede estar vacia")

        self.facultad = facultad.strip()

    def limite_prestamos(self):
        return 5

    def __str__(self):
        return (
            f"{super().__str__()} | "
            f"Facultad: {self.facultad}"
        )

6.**Clase Prestamo**

In [5]:
class Prestamo:
    def __init__(self, id_prestamo, material, usuario, fecha_inicio):
        if not isinstance(id_prestamo, str):
            raise TypeError("El ID del prestamo debe ser texto")

        if not isinstance(material, Material):
            raise TypeError("El material debe pertenecer a la clase Material")

        if not isinstance(usuario, Usuario):
            raise TypeError("El usuario debe pertenecer a la clase Usuario")

        if type(fecha_inicio) is not date:
            raise TypeError("La fecha de inicio debe ser un objeto date")

        self.id_prestamo = id_prestamo
        self.material = material
        self.usuario = usuario
        self.fecha_inicio = fecha_inicio

        self.fecha_prevista = (
            fecha_inicio + timedelta(days=material.dias_prestamo())
        )

        self.fecha_devolucion_real = None

    @property
    def activo(self):
        return self.fecha_devolucion_real is None

    def registrar_devolucion(self, fecha):
        if type(fecha) is not date:
            raise TypeError("La fecha debe ser un objeto date")

        if fecha < self.fecha_inicio:
            raise ValueError("La devolucion no puede ser anterior al inicio")

        if not self.activo:
            raise ValueError("El prestamo ya fue cerrado")

        self.fecha_devolucion_real = fecha

    def dias_atraso(self, fecha_consulta):
        if type(fecha_consulta) is not date:
            raise TypeError("La fecha de consulta debe ser un objeto date")

        if self.activo:
            if fecha_consulta < self.fecha_inicio:
                raise ValueError("La consulta no puede ser anterior al inicio")

            fecha_final = fecha_consulta
        else:
            fecha_final = self.fecha_devolucion_real

        atraso = (fecha_final - self.fecha_prevista).days

        return max(0, atraso)

    def __str__(self):
        estado = "Activo" if self.activo else "Cerrado"

        return (
            f"ID: {self.id_prestamo} | "
            f"Usuario: {self.usuario.nombre} | "
            f"Material: {self.material.titulo} | "
            f"Fecha de inicio: {self.fecha_inicio} | "
            f"Fecha prevista: {self.fecha_prevista} | "
            f"Fecha real: {self.fecha_devolucion_real} | "
            f"Estado: {estado}"
        )

7.**Clase Biblioteca**

In [6]:
class Biblioteca:
    def __init__(self, nombre):
        if not isinstance(nombre, str):
            raise TypeError("El nombre debe ser un texto")

        if not nombre.strip():
            raise ValueError("El nombre no puede estar vacio")

        self.nombre = nombre.strip()
        self.materiales = {}
        self.usuarios = {}
        self.prestamos = {}
        self.contador_prestamos = 0

    # 7.1 Registro y busqueda de materiales
    def agregar_material(self, material):
        if not isinstance(material, Material):
            raise TypeError("Debe registrar un material valido")

        codigo = material.codigo

        if codigo in self.materiales:
            raise ValueError("El codigo del material ya existe")

        self.materiales[codigo] = material

    # Registro de usuarios
    def registrar_usuario(self, usuario):
        if not isinstance(usuario, Usuario):
            raise TypeError("Debe registrar un usuario valido")

        identificacion = usuario.identificacion

        if identificacion in self.usuarios:
            raise ValueError("La identificacion ya existe")

        self.usuarios[identificacion] = usuario

    # Buscar material
    def buscar_material(self, codigo):
        if not isinstance(codigo, str):
            raise TypeError("El codigo debe ser texto")

        codigo = codigo.strip().upper()

        if codigo not in self.materiales:
            raise ValueError("Material no encontrado")

        return self.materiales[codigo]

    # Buscar usuario
    def buscar_usuario(self, identificacion):
        if not isinstance(identificacion, str):
            raise TypeError("La identificacion debe ser texto")

        identificacion = identificacion.strip()

        if identificacion not in self.usuarios:
            raise ValueError("Usuario no encontrado")

        return self.usuarios[identificacion]

    # 7.2 Registro de prestamos
    def prestar_material(self, codigo_material, identificacion_usuario, fecha_inicio):
        material = self.buscar_material(codigo_material)
        usuario = self.buscar_usuario(identificacion_usuario)

        if type(fecha_inicio) is not date:
            raise TypeError("La fecha debe ser un objeto date")

        if not material.disponible:
            raise ValueError("El material no esta disponible")

        prestamos_usuario = [
            p for p in self.prestamos.values()
            if p.usuario.identificacion == usuario.identificacion and p.activo
        ]

        if len(prestamos_usuario) >= usuario.limite_prestamos():
            raise ValueError("El usuario alcanzo su limite de prestamos")

        nuevo_id = f"PR{self.contador_prestamos + 1:03d}"

        nuevo_prestamo = Prestamo(
            nuevo_id,
            material,
            usuario,
            fecha_inicio
        )

        # Se modifica el estado despues de validar todas las condiciones
        material.prestar()

        self.prestamos[nuevo_id] = nuevo_prestamo
        self.contador_prestamos += 1

        return nuevo_prestamo

    # 7.3 Devolucion de materiales
    def devolver_material(self, id_prestamo, fecha_devolucion):
        if not isinstance(id_prestamo, str):
            raise TypeError("El ID del prestamo debe ser texto")

        if type(fecha_devolucion) is not date:
            raise TypeError("La fecha debe ser un objeto date")

        if id_prestamo not in self.prestamos:
            raise ValueError("Prestamo no encontrado")

        prestamo = self.prestamos[id_prestamo]

        if not prestamo.activo:
            raise ValueError("El prestamo ya fue devuelto")

        if fecha_devolucion < prestamo.fecha_inicio:
            raise ValueError("La devolucion no puede ser anterior al inicio")

        # Se valida y registra la devolucion
        prestamo.registrar_devolucion(fecha_devolucion)
        prestamo.material.devolver()

        return prestamo

    # 7.4 Consultas y reportes

    # Mostrar catalogo
    def mostrar_catalogo(self):
        for material in self.materiales.values():
            print(material)
            print(
                f"Plazo de prestamo: {material.dias_prestamo()} dias"
            )
            print("-" * 60)

    # Obtener materiales disponibles
    def materiales_disponibles(self):
        return [
            material for material in self.materiales.values()
            if material.disponible
        ]

    # Obtener prestamos activos
    def prestamos_activos(self):
        return [
            prestamo for prestamo in self.prestamos.values()
            if prestamo.activo
        ]

    # Obtener historial de un usuario
    def prestamos_de_usuario(self, identificacion):
        usuario = self.buscar_usuario(identificacion)

        return [
            prestamo for prestamo in self.prestamos.values()
            if prestamo.usuario.identificacion == usuario.identificacion
        ]

    # Obtener prestamos atrasados
    def prestamos_atrasados(self, fecha_consulta):
        if type(fecha_consulta) is not date:
            raise TypeError("La fecha debe ser un objeto date")

        return [
            prestamo for prestamo in self.prestamos.values()
            if prestamo.activo
            and prestamo.dias_atraso(fecha_consulta) > 0
        ]

    # Mostrar resumen
    def mostrar_resumen(self):
        activos = len(self.prestamos_activos())
        cerrados = len(self.prestamos) - activos

        print(f"Biblioteca: {self.nombre}")
        print(f"Materiales registrados: {len(self.materiales)}")
        print(f"Usuarios registrados: {len(self.usuarios)}")
        print(f"Prestamos activos: {activos}")
        print(f"Prestamos cerrados: {cerrados}")

    # 7.5 Metodos especiales
    def __len__(self):
        return len(self.materiales)

    def __str__(self):
        return (
            f"Biblioteca: {self.nombre} | "
            f"Materiales: {len(self.materiales)} | "
            f"Usuarios: {len(self.usuarios)} | "
            f"Prestamos: {len(self.prestamos)}"
        )

8.**Restricciones de diseño**

En el desarrollo del sistema se aplicaron las restricciones establecidas en la actividad:

1. Los plazos de los materiales se obtienen mediante el método dias_prestamo().

2. Los límites de préstamos se obtienen mediante el método limite_prestamos().

3. Se utiliza isinstance() para comprobar que los objetos pertenezcan a las jerarquías Material y Usuario.

4. La disponibilidad de los materiales se modifica únicamente mediante los métodos prestar() y devolver().

5. Los materiales, usuarios y préstamos se almacenan dentro de cada objeto Biblioteca, evitando el uso de variables globales.

6. Se utiliza TypeError cuando se recibe un tipo de dato incorrecto y ValueError cuando un valor o estado no está permitido.

7. Las operaciones se validan antes de modificar el historial o la disponibilidad de los materiales.

9.**Demostración obligatoria**


9.1 Crear la biblioteca y registrar los materiales

In [7]:
biblioteca = Biblioteca("Biblioteca Universitaria")

# Registrar tres libros
biblioteca.agregar_material(
    Libro("L001", "Python desde cero", "Juan Perez")
)

biblioteca.agregar_material(
    Libro("L002", "Programacion orientada a objetos", "Maria Lopez")
)

biblioteca.agregar_material(
    Libro("L003", "Inteligencia artificial", "Carlos Ruiz")
)

# Registrar dos revistas
biblioteca.agregar_material(
    Revista("R001", "Ciencia y Tecnologia", 15)
)

biblioteca.agregar_material(
    Revista("R002", "Innovacion Digital", 20)
)

# Registrar dos audiolibros
biblioteca.agregar_material(
    Audiolibro("A001", "Aprendiendo Python", 180)
)

biblioteca.agregar_material(
    Audiolibro("A002", "Introduccion a la IA", 240)
)

# Registrar dos estudiantes
biblioteca.registrar_usuario(
    Estudiante("EST001", "Pedro Gomez", "Ciencia de Datos")
)

biblioteca.registrar_usuario(
    Estudiante("EST002", "Ana Torres", "Ingenieria en Software")
)

# Registrar un docente
biblioteca.registrar_usuario(
    Docente("DOC001", "Luis Martinez", "Facultad de Ingenieria")
)

print("CATALOGO INICIAL")
biblioteca.mostrar_catalogo()

print("Cantidad total de materiales:", len(biblioteca))

CATALOGO INICIAL
Codigo: L001 | Titulo: Python desde cero | Estado: Disponible | Autor: Juan Perez | Plazo: 14 dias
Plazo de prestamo: 14 dias
------------------------------------------------------------
Codigo: L002 | Titulo: Programacion orientada a objetos | Estado: Disponible | Autor: Maria Lopez | Plazo: 14 dias
Plazo de prestamo: 14 dias
------------------------------------------------------------
Codigo: L003 | Titulo: Inteligencia artificial | Estado: Disponible | Autor: Carlos Ruiz | Plazo: 14 dias
Plazo de prestamo: 14 dias
------------------------------------------------------------
Codigo: R001 | Titulo: Ciencia y Tecnologia | Estado: Disponible | Edicion: 15 | Plazo: 7 dias
Plazo de prestamo: 7 dias
------------------------------------------------------------
Codigo: R002 | Titulo: Innovacion Digital | Estado: Disponible | Edicion: 20 | Plazo: 7 dias
Plazo de prestamo: 7 dias
------------------------------------------------------------
Codigo: A001 | Titulo: Aprendiendo Py

9.2 Registrar préstamos de distintos materiales

In [8]:
# Utilizar una fecha fija para la demostracion
fecha_inicio = date(2026, 11, 2)

# Registrar tres prestamos para el estudiante EST001
p1 = biblioteca.prestar_material(
    "L001", "EST001", fecha_inicio
)

p2 = biblioteca.prestar_material(
    "R001", "EST001", fecha_inicio
)

p3 = biblioteca.prestar_material(
    "A001", "EST001", fecha_inicio
)

# Registrar un prestamo para el docente
p4 = biblioteca.prestar_material(
    "L002", "DOC001", fecha_inicio
)

print("PRESTAMOS REGISTRADOS")

print(p1)
print(p2)
print(p3)
print(p4)

PRESTAMOS REGISTRADOS
ID: PR001 | Usuario: Pedro Gomez | Material: Python desde cero | Fecha de inicio: 2026-11-02 | Fecha prevista: 2026-11-16 | Fecha real: None | Estado: Activo
ID: PR002 | Usuario: Pedro Gomez | Material: Ciencia y Tecnologia | Fecha de inicio: 2026-11-02 | Fecha prevista: 2026-11-09 | Fecha real: None | Estado: Activo
ID: PR003 | Usuario: Pedro Gomez | Material: Aprendiendo Python | Fecha de inicio: 2026-11-02 | Fecha prevista: 2026-11-12 | Fecha real: None | Estado: Activo
ID: PR004 | Usuario: Luis Martinez | Material: Programacion orientada a objetos | Fecha de inicio: 2026-11-02 | Fecha prevista: 2026-11-16 | Fecha real: None | Estado: Activo


9.3 Consultar los préstamos de un estudiante

In [9]:
print("HISTORIAL DEL ESTUDIANTE EST001")

for prestamo in biblioteca.prestamos_de_usuario("EST001"):
    print(prestamo)

usuario = biblioteca.buscar_usuario("EST001")

print("Limite de prestamos:", usuario.limite_prestamos())

HISTORIAL DEL ESTUDIANTE EST001
ID: PR001 | Usuario: Pedro Gomez | Material: Python desde cero | Fecha de inicio: 2026-11-02 | Fecha prevista: 2026-11-16 | Fecha real: None | Estado: Activo
ID: PR002 | Usuario: Pedro Gomez | Material: Ciencia y Tecnologia | Fecha de inicio: 2026-11-02 | Fecha prevista: 2026-11-09 | Fecha real: None | Estado: Activo
ID: PR003 | Usuario: Pedro Gomez | Material: Aprendiendo Python | Fecha de inicio: 2026-11-02 | Fecha prevista: 2026-11-12 | Fecha real: None | Estado: Activo
Limite de prestamos: 3


9.4 Intentar registrar un cuarto préstamo

In [10]:
print("INTENTO DE REGISTRAR UN CUARTO PRESTAMO")

try:
    biblioteca.prestar_material(
        "R002",
        "EST001",
        date(2026, 11, 2)
    )

except ValueError as error:
    print("Operacion rechazada:", error)

INTENTO DE REGISTRAR UN CUARTO PRESTAMO
Operacion rechazada: El usuario alcanzo su limite de prestamos


9.5 Intentar prestar un material ocupado

In [11]:
print("INTENTO DE PRESTAR UN MATERIAL OCUPADO")

try:
    biblioteca.prestar_material(
        "R001",
        "EST002",
        date(2026, 11, 2)
    )

except ValueError as error:
    print("Operacion rechazada:", error)

print(
    "Disponibilidad de R001:",
    biblioteca.buscar_material("R001").disponible
)

INTENTO DE PRESTAR UN MATERIAL OCUPADO
Operacion rechazada: El material no esta disponible
Disponibilidad de R001: False


9.6 Registrar una devolución dentro del plazo y otra con retraso

In [13]:
# Devolucion dentro del plazo
biblioteca.devolver_material(
    "PR001",
    date(2026, 11, 10)
)

# Devolucion con retraso
biblioteca.devolver_material(
    "PR004",
    date(2026, 11, 25)
)

print("DEVOLUCIONES REGISTRADAS")

print("\nPrestamo PR001:")
print(biblioteca.prestamos["PR001"])

print(
    "Dias de atraso:",
    biblioteca.prestamos["PR001"].dias_atraso(date(2026, 11, 20))
)

print("\nPrestamo PR004:")
print(biblioteca.prestamos["PR004"])

print(
    "Dias de atraso:",
    biblioteca.prestamos["PR004"].dias_atraso(date(2026, 11, 25))
)

DEVOLUCIONES REGISTRADAS

Prestamo PR001:
ID: PR001 | Usuario: Pedro Gomez | Material: Python desde cero | Fecha de inicio: 2026-11-02 | Fecha prevista: 2026-11-16 | Fecha real: 2026-11-10 | Estado: Cerrado
Dias de atraso: 0

Prestamo PR004:
ID: PR004 | Usuario: Luis Martinez | Material: Programacion orientada a objetos | Fecha de inicio: 2026-11-02 | Fecha prevista: 2026-11-16 | Fecha real: 2026-11-25 | Estado: Cerrado
Dias de atraso: 9


9.7 Consultar préstamos activos atrasados e historial

In [14]:
print("PRESTAMOS ACTIVOS ATRASADOS AL 20 DE NOVIEMBRE")

fecha_consulta = date(2026, 11, 20)

atrasados = biblioteca.prestamos_atrasados(fecha_consulta)

for prestamo in atrasados:
    print(prestamo)

    print(
        "Dias de atraso:",
        prestamo.dias_atraso(fecha_consulta)
    )

print("\nHISTORIAL COMPLETO DEL ESTUDIANTE EST001")

for prestamo in biblioteca.prestamos_de_usuario("EST001"):
    print(prestamo)

PRESTAMOS ACTIVOS ATRASADOS AL 20 DE NOVIEMBRE
ID: PR002 | Usuario: Pedro Gomez | Material: Ciencia y Tecnologia | Fecha de inicio: 2026-11-02 | Fecha prevista: 2026-11-09 | Fecha real: None | Estado: Activo
Dias de atraso: 11
ID: PR003 | Usuario: Pedro Gomez | Material: Aprendiendo Python | Fecha de inicio: 2026-11-02 | Fecha prevista: 2026-11-12 | Fecha real: None | Estado: Activo
Dias de atraso: 8

HISTORIAL COMPLETO DEL ESTUDIANTE EST001
ID: PR001 | Usuario: Pedro Gomez | Material: Python desde cero | Fecha de inicio: 2026-11-02 | Fecha prevista: 2026-11-16 | Fecha real: 2026-11-10 | Estado: Cerrado
ID: PR002 | Usuario: Pedro Gomez | Material: Ciencia y Tecnologia | Fecha de inicio: 2026-11-02 | Fecha prevista: 2026-11-09 | Fecha real: None | Estado: Activo
ID: PR003 | Usuario: Pedro Gomez | Material: Aprendiendo Python | Fecha de inicio: 2026-11-02 | Fecha prevista: 2026-11-12 | Fecha real: None | Estado: Activo


9.8 Mostrar el resumen y crear una segunda biblioteca

In [15]:
print("RESUMEN DE LA BIBLIOTECA UNIVERSITARIA")

biblioteca.mostrar_resumen()

print("\nCREAR UNA SEGUNDA BIBLIOTECA")

biblioteca2 = Biblioteca("Biblioteca Secundaria")

print(biblioteca2)

print("Materiales:", len(biblioteca2))
print("Usuarios:", len(biblioteca2.usuarios))
print("Prestamos:", len(biblioteca2.prestamos))

RESUMEN DE LA BIBLIOTECA UNIVERSITARIA
Biblioteca: Biblioteca Universitaria
Materiales registrados: 7
Usuarios registrados: 3
Prestamos activos: 2
Prestamos cerrados: 2

CREAR UNA SEGUNDA BIBLIOTECA
Biblioteca: Biblioteca Secundaria | Materiales: 0 | Usuarios: 0 | Prestamos: 0
Materiales: 0
Usuarios: 0
Prestamos: 0


10.**Pruebas obligatorias**


En esta sección vamos a comprobar los errores esperados, los límites, los plazos, la normalización de códigos y la independencia de las bibliotecas.

Para que las pruebas no alteren la demostración anterior, vamos a crear una biblioteca exclusiva para realizar los ejercicios.

10.1 Crear biblioteca de pruebas

In [16]:
pruebas = Biblioteca("Biblioteca de Pruebas")

# Registrar materiales
pruebas.agregar_material(
    Libro("L101", "Libro de prueba", "Autor de prueba")
)

pruebas.agregar_material(
    Revista("R101", "Revista de prueba", 1)
)

pruebas.agregar_material(
    Audiolibro("A101", "Audiolibro de prueba", 120)
)

pruebas.agregar_material(
    Libro("L102", "Segundo libro", "Segundo autor")
)

# Registrar usuarios
pruebas.registrar_usuario(
    Estudiante("EST101", "Pedro", "Ciencia de Datos")
)

pruebas.registrar_usuario(
    Estudiante("EST102", "Ana", "Software")
)

pruebas.registrar_usuario(
    Docente("DOC101", "Luis", "Ingenieria")
)

print(pruebas)

Biblioteca: Biblioteca de Pruebas | Materiales: 4 | Usuarios: 3 | Prestamos: 0


10.2 Comprobar validaciones y errores

In [17]:
print("INICIO DE LAS PRUEBAS OBLIGATORIAS")

# PRUEBA 1: Titulo vacio
try:
    Libro("L110", "   ", "Autor")
except ValueError:
    print("Prueba 1 correcta: titulo vacio rechazado")

# PRUEBA 2: Numero de edicion igual a cero
try:
    Revista("R110", "Revista de prueba", 0)
except ValueError:
    print("Prueba 2 correcta: edicion cero rechazada")

# PRUEBA 3: Duracion negativa
try:
    Audiolibro("A110", "Audio de prueba", -20)
except ValueError:
    print("Prueba 3 correcta: duracion negativa rechazada")

# PRUEBA 4: Registrar un objeto que no sea material
try:
    pruebas.agregar_material("No soy un material")
except TypeError:
    print("Prueba 4 correcta: tipo incorrecto rechazado")

# PRUEBA 5: Codigo duplicado
try:
    pruebas.agregar_material(
        Libro("L101", "Otro libro", "Otro autor")
    )
except ValueError:
    print("Prueba 5 correcta: codigo duplicado rechazado")

# PRUEBA 6: Identificacion duplicada
try:
    pruebas.registrar_usuario(
        Estudiante("EST101", "Otro estudiante", "Informatica")
    )
except ValueError:
    print("Prueba 6 correcta: identificacion duplicada rechazada")

# PRUEBA 7: Buscar material inexistente
try:
    pruebas.buscar_material("NOEXISTE")
except ValueError:
    print("Prueba 7 correcta: material inexistente rechazado")

# PRUEBA 8: Plazos de los materiales
assert Libro("L120", "Libro", "Autor").dias_prestamo() == 14
assert Revista("R120", "Revista", 1).dias_prestamo() == 7
assert Audiolibro("A120", "Audio", 100).dias_prestamo() == 10

print("Prueba 8 correcta: plazos de materiales")

# PRUEBA 9: Limites de los usuarios
assert Estudiante("EST120", "Pedro", "Software").limite_prestamos() == 3
assert Docente("DOC120", "Luis", "Ingenieria").limite_prestamos() == 5

print("Prueba 9 correcta: limites de usuarios")

# PRUEBA 10: Normalizacion de codigos
material_normalizado = Libro(
    "  l130  ",
    "Libro normalizado",
    "Autor"
)

assert material_normalizado.codigo == "L130"

print("Prueba 10 correcta: normalizacion de codigo")

# PRUEBA 11: Superar el limite de prestamos
pruebas.prestar_material("L101", "EST101", date(2026, 11, 2))
pruebas.prestar_material("R101", "EST101", date(2026, 11, 2))
pruebas.prestar_material("A101", "EST101", date(2026, 11, 2))

try:
    pruebas.prestar_material("L102", "EST101", date(2026, 11, 2))
except ValueError:
    print("Prueba 11 correcta: limite de prestamos alcanzado")

# PRUEBA 12: Prestar un material ocupado
try:
    pruebas.prestar_material("L101", "EST102", date(2026, 11, 2))
except ValueError:
    print("Prueba 12 correcta: material ocupado rechazado")

# PRUEBA 13: Devolucion anterior al inicio
try:
    pruebas.devolver_material("PR001", date(2026, 11, 1))
except ValueError:
    print("Prueba 13 correcta: devolucion anterior rechazada")

# PRUEBA 14: Consultar antes de la fecha de inicio
try:
    pruebas.prestamos["PR001"].dias_atraso(date(2026, 11, 1))
except ValueError:
    print("Prueba 14 correcta: consulta anterior rechazada")

# PRUEBA 15: Devolucion correcta
pruebas.devolver_material("PR001", date(2026, 11, 16))

assert pruebas.prestamos["PR001"].activo is False
assert pruebas.buscar_material("L101").disponible is True

print("Prueba 15 correcta: devolucion registrada")

# PRUEBA 16: Devolver dos veces
try:
    pruebas.devolver_material("PR001", date(2026, 11, 17))
except ValueError:
    print("Prueba 16 correcta: doble devolucion rechazada")

# PRUEBA 17: Atraso cero al devolver en la fecha prevista
assert pruebas.prestamos["PR001"].dias_atraso(date(2026, 11, 20)) == 0

print("Prueba 17 correcta: atraso cero en la fecha prevista")

# PRUEBA 18: Bibliotecas independientes
biblioteca_vacia = Biblioteca("Biblioteca Independiente")

assert len(biblioteca_vacia) == 0
assert len(biblioteca_vacia.usuarios) == 0
assert len(biblioteca_vacia.prestamos) == 0

print("Prueba 18 correcta: bibliotecas independientes")

print("\nTODAS LAS PRUEBAS FINALIZARON CORRECTAMENTE")

INICIO DE LAS PRUEBAS OBLIGATORIAS
Prueba 1 correcta: titulo vacio rechazado
Prueba 2 correcta: edicion cero rechazada
Prueba 3 correcta: duracion negativa rechazada
Prueba 4 correcta: tipo incorrecto rechazado
Prueba 5 correcta: codigo duplicado rechazado
Prueba 6 correcta: identificacion duplicada rechazada
Prueba 7 correcta: material inexistente rechazado
Prueba 8 correcta: plazos de materiales
Prueba 9 correcta: limites de usuarios
Prueba 10 correcta: normalizacion de codigo
Prueba 11 correcta: limite de prestamos alcanzado
Prueba 12 correcta: material ocupado rechazado
Prueba 13 correcta: devolucion anterior rechazada
Prueba 14 correcta: consulta anterior rechazada
Prueba 15 correcta: devolucion registrada
Prueba 16 correcta: doble devolucion rechazada
Prueba 17 correcta: atraso cero en la fecha prevista
Prueba 18 correcta: bibliotecas independientes

TODAS LAS PRUEBAS FINALIZARON CORRECTAMENTE


10.3 Comprobar que los préstamos rechazados no modifican los datos

In [19]:
print("PRUEBA DE CONSISTENCIA DE DATOS")

# Guardar el estado antes de intentar el prestamo
cantidad_prestamos_antes = len(pruebas.prestamos)

disponibilidad_antes = pruebas.buscar_material("L101").disponible

try:
    # L101 ya se encuentra prestado
    pruebas.prestar_material(
        "L101",
        "EST102",
        date(2026, 11, 2)
    )

except ValueError:
    print("El sistema rechazo correctamente el prestamo")

# Comprobar que el historial no cambio
assert len(pruebas.prestamos) == cantidad_prestamos_antes

# Comprobar que la disponibilidad no cambio
assert pruebas.buscar_material("L101").disponible == disponibilidad_antes

print("El historial se mantiene sin cambios")
print("La disponibilidad se mantiene sin cambios")
print("Prueba de consistencia completada correctamente")

PRUEBA DE CONSISTENCIA DE DATOS
El sistema rechazo correctamente el prestamo
El historial se mantiene sin cambios
La disponibilidad se mantiene sin cambios
Prueba de consistencia completada correctamente


11.**Preguntas de análisis**

1. **¿Qué estado interno protege el encapsulamiento en tu solución?**

El encapsulamiento protege principalmente la disponibilidad de los materiales y la identificación de los usuarios. Estos datos se almacenan como atributos internos y se consultan mediante propiedades de solo lectura. Esto evita que sean modificados directamente y permite mantener un mejor control de la información.

2.**¿Qué código reutilizan las clases hijas mediante super()?**

Las clases Libro, Revista y Audiolibro reutilizan el constructor de Material mediante super(). De igual manera, las clases Estudiante y Docente reutilizan el constructor de Usuario. Esto permite aprovechar las validaciones y evitar repetir código.

3.**¿Dónde aplicaste polimorfismo en la gestión de materiales?**

El polimorfismo se aplica mediante el método dias_prestamo(). Cada tipo de material tiene un plazo diferente: los libros tienen 14 días, las revistas 7 días y los audiolibros 10 días. La biblioteca utiliza el mismo método sin necesidad de identificar el tipo concreto de material.

4. **¿Dónde aplicaste polimorfismo en la gestión de usuarios?**

Se aplica mediante el método limite_prestamos(). Los estudiantes pueden mantener hasta tres préstamos activos y los docentes hasta cinco. La biblioteca consulta este método sin distinguir directamente entre las clases de usuario.

5.**¿Qué diferencia existe entre heredar de una clase y guardar un objeto de otra clase como atributo? Utiliza Libro y Prestamo como ejemplos.**

Heredar significa que una clase recibe características y métodos de otra clase. Por ejemplo, Libro hereda de Material. En cambio, guardar un objeto como atributo significa utilizarlo dentro de otra clase. Por ejemplo, Prestamo contiene un objeto Material y un objeto Usuario para representar la información del préstamo.

6. **¿Por qué conviene validar todas las condiciones antes de registrar un préstamo?**

Porque permite evitar errores y mantener la información consistente. Si un material está ocupado o un usuario alcanzó su límite, la operación debe rechazarse sin modificar la disponibilidad del material ni el historial de préstamos.

7. **¿Qué cambios serían necesarios para incorporar un Investigador con un límite de ocho préstamos?**

Se debe crear una nueva clase Investigador que herede de Usuario, agregar los atributos específicos que necesite y sobrescribir el método limite_prestamos() para que devuelva 8. No sería necesario modificar la lógica principal de la biblioteca.

8. **¿Qué cambios serían necesarios para incorporar un VideoEducativo con un plazo de cinco días?**

Se debe crear una clase VideoEducativo que herede de Material y sobrescriba el método dias_prestamo() para que devuelva 5. También se podrían agregar atributos específicos para almacenar información del video.

9. **¿Por qué cada biblioteca necesita colecciones independientes?**

Porque cada biblioteca debe administrar sus propios materiales, usuarios y préstamos. De esta manera, los registros de una biblioteca no afectan a otra y se mantiene la información organizada de forma independiente.

10. **¿Qué aportan __str__() y __len__() a la forma de utilizar tus objetos?**

El método __str__() permite mostrar la información de los objetos de una manera clara y legible. El método __len__() permite conocer la cantidad de materiales registrados utilizando directamente la función len().

12. **Conclusiones y bibliografía**

Durante el desarrollo de esta actividad aprendí a utilizar la programación orientada a objetos en Python mediante clases, objetos, herencia, encapsulamiento y polimorfismo.

También comprendí cómo utilizar super() para reutilizar código y cómo sobrescribir métodos para establecer diferentes comportamientos en cada clase. Esto permitió organizar los materiales y usuarios de una manera más clara.

Una de las principales dificultades fue controlar correctamente los préstamos y las devoluciones, especialmente al validar las fechas, la disponibilidad de los materiales y los límites de préstamos de cada usuario. Esta dificultad se resolvió mediante validaciones y el manejo de excepciones utilizando TypeError y ValueError.

Finalmente, comprendí la importancia de mantener los datos organizados, evitar modificaciones incorrectas y comprobar el funcionamiento del programa mediante pruebas con assert.

**BIBLIOGRAFÍA**

Python Software Foundation. Clases:
https://docs.python.org/3/tutorial/classes.html

Python Software Foundation. Modelo de datos:
https://docs.python.org/3/reference/datamodel.html

Python Software Foundation. datetime:
https://docs.python.org/3/library/datetime.html